# Chapitre 1_Theoreme_fondamental_d_evaluation_des_actifs

Avant de valoriser un produit dérivé, il faut comprendre **pourquoi** on actualise 
au taux sans risque plutôt qu'au rendement espéré.  
Ce chapitre pose ce cadre théorique, puis implémente les outils concrets qui en découlent.


---


### Le théorème en une phrase


Dans un marché $\mathcal{M}$, les trois affirmations suivantes sont équivalentes :


- Il n'existe **aucune opportunité d'arbitrage**
- Il existe une **mesure martingale** $\mathbb{Q}$ équivalente à $P$
- Le système de prix est **linéaire et cohérent**


La conséquence directe : si le marché est sans arbitrage, il existe un prix unique 
$V_0$ pour tout produit dérivé $V_T$ :


$$
\forall \mathbb{Q} : V_0 = \mathbb{E}^{\mathbb{Q}}\left[e^{-rT} V_T\right]
$$


On remplace la probabilité réelle $P$ par $\mathbb{Q}$, sous laquelle tous les 
actifs risqués dérivent au taux sans risque $r$.


> 📌 **À savoir**  
> Un marché est dit *complet* s'il est sans arbitrage **et** si tout produit dérivé 
> est réplicable — ce qui équivaut à l'unicité de $\mathbb{Q}$.  
>
> Dans le cas général, le modèle de marché $\mathcal{M}$ est défini formellement par 
> $\{\Omega,\ \wp(\Omega),\ \mathbb{F},\ P,\ T,\ \mathbb{S}\}$ :
> espace d'états, tribu, filtration, mesure de probabilité, horizon et processus de prix.


---


### Pourquoi $\mathbb{Q}$ et pas $P$ ?


Un stock vaut 10 aujourd'hui. Demain : 20 (proba 60%) ou 0 (proba 40%).  
Une option call de strike 15 paie 5 si le stock monte, 0 sinon.


Sous $P$ (probabilité réelle) :

$$
\mathbb{E}^P[V_T] = 0{,}6 \times 5 + 0{,}4 \times 0 = 3 \text{ USD}
$$

Mais le prix de réplication correct est **2,5 USD** — $P$ surestime l'option 
car elle intègre la prime de risque du stock.

Sous $\mathbb{Q}$ (probabilité risque-neutre, 50/50) :

$$
\mathbb{E}^{\mathbb{Q}}[V_T] = 0{,}5 \times 5 + 0{,}5 \times 0 = 2{,}5 \text{ USD} \checkmark
$$

Ce changement de probabilité est exactement ce que la mesure martingale $\mathbb{Q}$ réalise : 
elle retire la prime de risque pour ne garder que la valeur de réplication.


---


### Taux court constant et actualisation


On suppose un **taux court $r$ constant** sur toute la durée de vie du contrat.  
La valeur actuelle d'un flux futur $V_T$ est alors :

$$
V_0 = e^{-rT} \cdot \mathbb{E}^{\mathbb{Q}}[V_T]
$$

La classe `ConstantShortRate` implémente ce facteur d'actualisation à partir 
d'une liste de dates, converties en fractions d'année via `get_year_deltas()` :


$$
\Delta_i = \frac{(\text{date}_i - \text{date}_0)\ \text{en jours}}{365}
$$


---


### Market Environment


Pour utiliser cette formule en pratique, on organise tous les paramètres nécessaires 
dans un objet unique appelé *market environment*.  
Il regroupe trois types d'éléments :


- **constants** — scalaires : taux $r$, volatilité $\sigma$, prix spot $S_0$…
- **lists** — grilles de dates, séries temporelles
- **curves** — objets d'actualisation (`ConstantShortRate`)


C'est le conteneur universel transmis à toutes les classes de simulation 
et de valorisation qui suivent dans les prochains chapitres.

<br>

# Chapitre 2_Actualisation_risque_neutre 

## Modélisation et traitement des dates, taux court constant 

Ce chapitre met en place l’infrastructure nécessaire pour évaluer des produits dérivés (options, futures, etc.).  
Avant de calculer un prix, il faut savoir **gérer le temps** autrement dit, représenter des dates et des durées en Python.

---

### comment représenter le temps ?

Pour valoriser un dérivé, on découpe la vie du contrat d’aujourd’hui jusqu’à la maturité $(T)$ en intervalles. 
Deux approches sont courantes :

- **Dates explicites** — ex. `2020-01-01`, `2020-07-01`, `2021-01-01`
- **Fractions d’année** — ex. `0.0`, `0.5`, `1.0` (où `0` = début, `1` = fin d’une année)

Les deux approches sont équivalentes, mais les fractions d’année sont souvent plus pratiques pour les formules.  
La première date devient toujours `0.0` : c’est le point de départ **normalisé**.

---

### La fonction `get_year_deltas()`

Cette fonction convertit une liste de dates en fractions d’année :

$
\Delta_i = \frac{(\text{date}_i - \text{date}_0)\ \text{en jours}}{365}
$

**Exemple :**  
`[2020-01-01, 2020-07-01, 2021-01-01]` → `[0.0, 0.497, 1.0]`

---

### Pourquoi « risque neutre » ?

L’actualisation en mesure risque-neutre est une base de la valorisation des dérivés.  
On calcule la valeur actuelle d’un flux futur en actualisant non pas au rendement espéré, mais au **taux sans risque** $(r)$, sous une probabilité artificielle (dite « risque neutre ») garantissant l’absence d’arbitrage :

$
V_0 = e^{-rT} \cdot \mathbb{E}^Q\left[V_T\right]
$

Ce chapitre pose donc les briques de base (gestion du temps) avant d’implémenter cette formule.

<br>

# Chapitre 3_Environnement_de_marche

## Un conteneur pour la valorisation

Ce chapitre introduit la classe `market_environment`, un objet qui regroupe
toutes les données nécessaires à la valorisation d'un dérivé.  
L'idée est simple : plutôt que de passer chaque paramètre séparément à chaque
classe, on les rassemble en un seul objet transmis en une seule fois.


---


### Pourquoi un environnement de marché ?


Dans les chapitres suivants, chaque simulation et chaque valorisation
auront besoin des mêmes informations : taux sans risque, volatilité,
prix initial, maturité, courbe d'actualisation...

Sans conteneur, on devrait passer tous ces paramètres un par un à chaque classe.  
Avec `market_environment`, on les regroupe **une seule fois** et on transmet
l'objet entier là où on en a besoin.


---


### Structure de l'environnement


Un environnement de marché se compose de trois dictionnaires :


- **constants** — les scalaires : taux $r$, volatilité $\sigma$, prix spot $S_0$, maturité...
- **lists** — les collections d'objets : sous-jacents, grilles de dates...
- **curves** — les courbes de marché : instance de `constant_short_rate`


---


### Ce qu'on implémente


La classe `market_environment` avec des méthodes `add_*` et `get_*`
pour chaque compartiment, ainsi qu'une méthode `add_environment`
qui permet de **fusionner deux environnements** en un seul.


> 📌 **À savoir**  
> Cette flexibilité a un revers : aucune vérification n'est faite sur
> les données passées à l'environnement. Dans un contexte de production,
> il faudrait ajouter des contrôles pour éviter des paramètres incohérents.
